In [ ]:
!pip install mtcnn mediapipe


## MTCNN

In [ ]:
import cv2
import random
from pathlib import Path
from mtcnn.mtcnn import MTCNN

from display_notebook import display_images_grid_html

# ランダムに画像(image_extに指定された拡張子の画像)を選択
image_ext = [".jpg", ".jpeg", ".png", ".bmp", ".gif", "webp"]
image_path = random.choice(list(Path("./sample_face/").glob("**/*")))
while image_path.is_dir() or image_path.suffix.lower() not in image_ext:
    image_path = random.choice(list(Path("./sample_face").glob("**/*")))
print(image_path)

# OpenCVで画像を読み込み、RGB形式に変換
image = cv2.imread(image_path)
if image is None:
    raise ValueError(f"{image_path} の画像が読み込めませんでした。")
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# MTCNNの検出器を初期化
detector = MTCNN()

# 画像から顔を検出
faces = detector.detect_faces(image_rgb)

# 検出された顔に対してバウンディングボックスと信頼度を描画
result_image = image_rgb.copy()
for face in faces:
    # face['box'] は [x, y, width, height] の形式
    x, y, width, height = face['box']
    cv2.rectangle(result_image, (x, y), (x + width, y + height), (0, 255, 0), 2)
    # 信頼度を描画（%表示）
    confidence = face['confidence'] * 100
    cv2.putText(result_image, f"{confidence:.1f}%", (x, y - 10),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36, 255, 12), 2)
    

display_images_grid_html(images=[image_rgb, result_image], labels=["元画像", "MTCNNによる顔検出"])

result_face_images = []
confidences = []
for face in faces:
    x, y, width, height = face['box']
    confidences.append(face['confidence'])
    result_face_images.append(image_rgb[y:y + height, x:x + width])

display_images_grid_html(images=result_face_images, labels=[f"face {i} : {confidences[i]:.05}" for i in range(len(result_face_images))], cols=10)



In [ ]:
# 実行時間の計測
import time
from tqdm import tqdm
from pathlib import Path
from mtcnn.mtcnn import MTCNN
from display_notebook import display_images_grid_html

test_num = 100
image_path = Path("./sample_face/kageyama_yuka/4.jpg")
org_image = cv2.imread(str(image_path))[:,:,::-1].copy()

# リサイズする
# image = cv2.resize(image, (256, 256))
raito = 0.1
image = cv2.resize(org_image, dsize=None, fx=raito, fy=raito)
print("Shape: ", image.shape)

display_images_grid_html(images=[org_image, image], labels=["元画像", "リサイズ後"])

# MTCNNの検出器を初期化
detector = MTCNN()

start = time.time()
detect_count = 0
for img_path in tqdm(range(test_num)):
    faces = detector.detect_faces(image)
    detect_count += len(faces)
end = time.time()

print(f"{test_num}回の検索にかかった時間: {end - start:.3f} 秒")
print(f"1回あたりの平均時間: {((end - start) / test_num):.8f} 秒")
print(f"検出された顔の数: {detect_count} / {test_num}")


## Mediapipe

In [ ]:
import cv2
import random
import mediapipe as mp
from pathlib import Path

from display_notebook import display_images_grid_html

# ランダムに画像(image_extに指定された拡張子の画像)を選択
image_ext = [".jpg", ".jpeg", ".png", ".bmp", ".gif", "webp"]
image_path = random.choice(list(Path("./sample_face/").glob("**/*")))
while image_path.is_dir() or image_path.suffix.lower() not in image_ext:
    image_path = random.choice(list(Path("./sample_face").glob("**/*")))
print(image_path)

# OpenCVで画像を読み込み、RGB形式に変換
image = cv2.imread(image_path)
if image is None:
    raise ValueError(f"{image_path} の画像が読み込めませんでした。")
image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

# MediapipeのFace Detectionモジュールを初期化
mp_face_detection = mp.solutions.face_detection
face_detection = mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.5)

# 顔検出を実行
results = face_detection.process(image_rgb)

# 画像のサイズを取得
height, width, _ = image.shape

# 検出結果がある場合、顔のバウンディングボックスを描画
result_image = image_rgb.copy()
if results.detections:
    for detection in results.detections:
        # relative_bounding_boxは画像サイズに対する相対値なので絶対値に変換
        bbox = detection.location_data.relative_bounding_box
        x_min = int(bbox.xmin * width)
        y_min = int(bbox.ymin * height)
        box_width = int(bbox.width * width)
        box_height = int(bbox.height * height)
        # 顔の検出確信度も描画（任意）
        score = detection.score[0]
        cv2.rectangle(result_image, (x_min, y_min), (x_min + box_width, y_min + box_height), (0, 255, 0), 2)
        cv2.putText(result_image, f"{score:.2f}", (x_min, y_min - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (36, 255, 12), 2)
        

display_images_grid_html(images=[image_rgb, result_image], labels=["元画像", "Mediapipeによる顔検出"])

if results.detections:
    result_face_images = []
    confidences = []
    for i, detection in enumerate(results.detections):
        bbox = detection.location_data.relative_bounding_box
        x_min = int(bbox.xmin * width)
        y_min = int(bbox.ymin * height)
        box_width = int(bbox.width * width)
        box_height = int(bbox.height * height)
        result_face_images.append(image_rgb[y_min:y_min + box_height, x_min:x_min + box_width])
        confidences.append(detection.score[0])

    display_images_grid_html(images=result_face_images, labels=[f"face {i} : {confidences[i]:.05}" for i in range(len(result_face_images))], cols=10)




In [ ]:
# 実行時間の計測
import time
from tqdm import tqdm
from pathlib import Path
import mediapipe as mp
from display_notebook import display_images_grid_html

test_num = 100
image_path = Path("./sample_face/kageyama_yuka/3.jpg")
org_image = cv2.imread(str(image_path))[:,:,::-1].copy()

# リサイズする
# image = cv2.resize(image, (256, 256))
raito = 1
image = cv2.resize(org_image, dsize=None, fx=raito, fy=raito)
print("Shape: ", image.shape)

display_images_grid_html(images=[org_image, image], labels=["元画像", "リサイズ後"])

# MediapipeのFace Detectionモジュールを初期化
mp_face_detection = mp.solutions.face_detection
face_detection = mp_face_detection.FaceDetection(model_selection=1, min_detection_confidence=0.5)

start = time.time()
detect_count = 0
for img_path in tqdm(range(test_num)):
    results = face_detection.process(image_rgb)
    detect_count += len(results.detections) if results.detections else 0
end = time.time()

print(f"{test_num}回の検索にかかった時間: {end - start:.3f} 秒")
print(f"1回あたりの平均時間: {((end - start) / test_num):.8f} 秒")
print(f"検出された顔の数: {detect_count} / {test_num}")
